# Particle-1 -> LFM2-1.2B QLoRA (turnkey)

Kaggle: Settings -> Accelerator **GPU T4 x2**, Internet **On**, then Run All.

Output: `/kaggle/working/particle1-lfm2-1.2b-Q4_0.gguf` (download from the Output panel).

No external accounts, no API keys. Training data is pulled from the public repo.

Priority domains: physics, sysadmin, systems engineering, coding, maths.
Style: reason when needed, shortest correct solution, no over-engineering.

In [ ]:
!pip -q install --upgrade transformers peft bitsandbytes accelerate datasets trl
import torch, transformers, peft, trl
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('transformers', transformers.__version__, '| peft', peft.__version__, '| trl', trl.__version__)

In [ ]:
!wget -q https://raw.githubusercontent.com/saksham-991/xiaomi-pad7-npu-llm/main/distill/data/train.jsonl -O /kaggle/working/train.jsonl
!wget -q https://raw.githubusercontent.com/saksham-991/xiaomi-pad7-npu-llm/main/distill/data/valid.jsonl -O /kaggle/working/valid.jsonl
import json
rows = [json.loads(l) for l in open('/kaggle/working/train.jsonl') if l.strip()]
print(len(rows), 'train examples')
print(rows[0]['messages'][1]['content'][:120])

In [ ]:
MODEL = 'LiquidAI/LFM2-1.2B'
MAXLEN = 2048
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                             device_map='auto', trust_remote_code=True)
model.config.use_cache = False
print('model loaded')


In [ ]:
from datasets import Dataset
def encode(ex):
    msgs = ex['messages']
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    answer = msgs[-1]['content']
    p = tok(prompt, add_special_tokens=False)['input_ids']
    a = tok(answer + (tok.eos_token or ''), add_special_tokens=False)['input_ids']
    ids = (p + a)[:MAXLEN]
    labels = ([-100] * len(p) + a)[:MAXLEN]
    return {'input_ids': ids, 'labels': labels, 'attention_mask': [1]*len(ids)}
ds = Dataset.from_list(rows).map(encode, remove_columns=['messages','domain'])
print(ds)
print('avg tokens', sum(len(x) for x in ds['input_ids'])/len(ds))

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
args = TrainingArguments(
    output_dir='/kaggle/working/ckpt',
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    num_train_epochs=2, learning_rate=1e-4, warmup_steps=5,
    lr_scheduler_type='cosine', logging_steps=5, save_strategy='no',
    fp16=True, bf16=False, report_to='none', seed=1337,
)
coll = DataCollatorForSeq2Seq(tokenizer=tok, padding=True, pad_to_multiple_of=8)
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=coll)
trainer.train()
trainer.save_model('/kaggle/working/adapter')
tok.save_pretrained('/kaggle/working/adapter')


In [ ]:
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
merged = PeftModel.from_pretrained(base, '/kaggle/working/adapter').merge_and_unload()
merged.save_pretrained('/kaggle/working/merged', safe_serialization=True)
tok.save_pretrained('/kaggle/working/merged')
print('merged saved')


In [ ]:
!git clone -q https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
!cd /kaggle/working/llama.cpp && git checkout -q b54752df40e4d8e53d34ef5a598336b4e4871d39
!pip -q install -r /kaggle/working/llama.cpp/requirements.txt
!cd /kaggle/working/llama.cpp && cmake -B build -DGGML_NATIVE=ON -DLLAMA_CURL=OFF -DGGML_OPENMP=ON >/dev/null && cmake --build build -j4 --target llama-quantize >/dev/null
print('llama.cpp ready')

In [ ]:
!python3 /kaggle/working/llama.cpp/convert_hf_to_gguf.py /kaggle/working/merged --outtype f16 --outfile /kaggle/working/model-f16.gguf
!/kaggle/working/llama.cpp/build/bin/llama-quantize /kaggle/working/model-f16.gguf /kaggle/working/particle1-lfm2-1.2b-Q4_0.gguf Q4_0
!rm -f /kaggle/working/model-f16.gguf
!ls -lh /kaggle/working/particle1-lfm2-1.2b-Q4_0.gguf